Setup

In [ ]:
!pip install -U transformers datasets evaluate accelerate
!pip install scikit-learn
!pip install tensorboard

Imports

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)

import evaluate
import glob
import numpy as np

Hyperparameters

In [ ]:
BATCH_SIZE = 100
NUM_PROCS = 32
LR = 0.00005
EPOCHS = 5
MODEL = 'dccuchile/bert-base-spanish-wwm-cased'
OUT_DIR = 'clickbait_bert'

Download the Dataset

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset


df = pd.read_csv("train_final_clickbait_es.csv")

# Definir la proporción de datos para cada conjunto
train_size = 0.7
val_size = 0.15
test_size = 0.15

# Dividir el dataset en train y test
X_train, X_test, y_train, y_test = train_test_split(df.text, df.label, test_size=test_size, random_state=42)

# Dividir el conjunto de train en train y validación
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=val_size/train_size, random_state=42)

train_dataset = pd.DataFrame({"text": X_train, "label": y_train})
valid_dataset = pd.DataFrame({"text": X_val, "label": y_val})
test_dataset = pd.DataFrame({"text": X_test, "label": y_test})

train_dataset = Dataset.from_pandas(train_dataset)
valid_dataset = Dataset.from_pandas(valid_dataset)
test_dataset = Dataset.from_pandas(test_dataset)



In [ ]:
# train_dataset = load_dataset("jpancorbTaniwa/train_final_clickbait_es", split='train')
# valid_dataset = load_dataset("jpancorbTaniwa/train_final_clickbait_es", split='validation')
# test_dataset = load_dataset("jpancorbTaniwa/train_final_clickbait_es", split='test')

In [ ]:
# Visualize a sample.
train_dataset[0]

Dataset Information

In [ ]:
id2label = {
    0: "NO Clickbait",
    1: "Clickbait"
}
label2id = {
    "NO Clickbait": 0,
    "Clickbait": 1
}

Tokenize the Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [ ]:
# Helper function for preprocessing.
def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
    )

In [ ]:
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=BATCH_SIZE,
    num_proc=NUM_PROCS
)

In [ ]:
tokenized_valid = valid_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=BATCH_SIZE,
    num_proc=NUM_PROCS
)

In [ ]:
tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=BATCH_SIZE,
    num_proc=NUM_PROCS
)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Sample Tokenization Example

In [ ]:
tokenized_sample = preprocess_function(train_dataset[0])

In [ ]:
print(tokenized_sample)
print(f"Length of tokenized IDs: {len(tokenized_sample.input_ids)}")
print(f"Length of attention mask: {len(tokenized_sample.attention_mask)}")

Evaluation Metrics

In [ ]:
accuracy = evaluate.load('accuracy')

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
# Total parameters and trainable parameters.
total_params = sum(p.numel() for p in model.parameters())
print(f"{total_params:,} total parameters.")
total_trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad)
print(f"{total_trainable_params:,} training parameters.")

Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=3,
    report_to='tensorboard',
    fp16=True
)

Training

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
history = trainer.train()

Evaluate

In [ ]:
trainer.evaluate(tokenized_test)

 Inference

In [ ]:
print(history.global_step)

Inference

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(f"clickbait_es/checkpoint-last")

access_token='***************************'

model.push_to_hub("jpancorbTaniwa/clickbait_es",token=access_token)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('clickbait_es/checkpoint-last')

access_token='*****************'

tokenizer.push_to_hub("YOURUSER/clickbait_es",token=access_token)


In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TextClassificationPipeline,
)

tokenizerDOS = AutoTokenizer.from_pretrained("taniwasl/clickbait_es")
modelDOS = AutoModelForSequenceClassification.from_pretrained("taniwasl/clickbait_es")

tokenizerDOS.save_pretrained('.')
torch.save(modelDOS.state_dict(), 'pytorch_model.bin')

review_text = 'La explosión destruye parcialmente el edificio, Egipto'

nlp = TextClassificationPipeline(task = "text-classification",
                model = modelDOS,
                tokenizer = tokenizerDOS,
                max_length = 25,
                truncation=True,
                add_special_tokens=True
                )

print(nlp(review_text))

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
# classify = pipeline(task='text-classification', model=model, tokenizer=tokenizer)

In [ ]:
# all_files = glob.glob('inference_data/*')
# for file_name in all_files:
#     file = open(file_name)
#     content = file.read()
#     print(content)
#     result = classify(content)
#     print('PRED: ', result)
#     print('GT: ', file_name.split('_')[-1].split('.txt')[0])
#     print('\n')